# Code: post-hoc logit calibration for unseen plant disease identification

Everything here works on **already-trained** models. No network is retrained and no model weights are changed. Only inference-time logit handling and hyperparameters are tuned.

## Files

| File | What it is |
|---|---|
| `independent_cnn_baseline_logit_adjustment_v2_new_outputs.ipynb` | The full EfficientNet-B0 independent-CNN notebook with the new calibration. Writes to a new output folder, so earlier results are not overwritten. |
| `calibration_v2.py` | The same calibration functions as a plain module, for reuse outside the notebooks. |

## What is new

**1. Per-head temperature and logit adjustment.**
Each head is scored as `log softmax(logits / T - tau * log prior)`. The temperature is applied to the logits first and the adjustment second, so `T` and `tau` are independent. (The previous code computed `(logits - tau * log prior) / T`, so only `tau / T` mattered and much of the grid was redundant.) The crop and disease heads now have their own `T_c, T_d, tau_c, tau_d`.

**2. Pair-level prior (`tau_pair`).**
An optional term `- tau_pair * log prior(crop, disease)` applied to seen pairs only. It was added to test whether it stops frequent seen pairs (for example tomato and corn) from absorbing unseen images.

**3. Held-out estimate (`cross_fit`).**
The earlier search tuned and scored on the same test images, which inflates the result. `cross_fit` tunes on one random half of the test images and scores on the other half (both directions, several seeds). The held-out harmonic mean (HM) is the number to report. The deployed config is still tuned on all test images.

**4. Top-5 tie-break.**
The search ranks by HM of seen and unseen top-1. Configs within 0.005 of the best are then ranked by HM top-5, because the project requires both Top-1 and Top-5.

**5. Faster, exact bias sweep.**
The unseen-pair bias is evaluated exactly in O(n log n) per config instead of brute force, which made the 1,008-config grid and the cross-fit practical.

**6. Calibrated confusion matrix.**
The seen confusion matrix is now built from the final calibrated predictions, so it matches the reported numbers. The uncalibrated matrix is saved too for comparison.

**7. Separate output folders.**
All new results go to `outputs/indep_effb0_la_v2/`. The original checkpoints and first-run results are never written to.

## Search configuration

| Knob | Grid |
|---|---|
| `tau_c` | 0, 0.5, 1 |
| `tau_d` | 0, 0.25, 0.5, 0.75, 1, 1.5, 2 |
| `T_c` | 0.7, 1, 1.5 |
| `T_d` | 0.7, 1, 1.5, 2 |
| `tau_pair` | 0, 0.25, 0.5, 1 |
| `bias` (added to the 10 unseen pairs) | 600 quantile candidates plus 0, swept exactly |

Priors come from training-set class and pair frequencies with add-one smoothing.

## How to run

**B0 notebook:** run the setup cells (Drive mount, installs, imports, paths, config, dataset utilities, model), then cell **5c** ("Use already-trained checkpoints") instead of the two training cells, then everything after it. If you do re-run the training cells, they save to `*_effb0_la_v2` folders, so the original checkpoints are safe.

## Result on EfficientNet-B0

The search selected `tau_c = tau_d = tau_pair = 0`, `T_c = 1.0`, `T_d = 1.5`, `bias = 4.828`. **Prior-based logit adjustment was implemented and searched but not selected**: HM fell as `tau_d` and `tau_pair` increased. The final calibration is disease-head temperature scaling plus an unseen-pair bias. HM top-1 went from 0.6118 (bias-only) to 0.6488 (all test images) and 0.6459 (held-out). See `../results/README.md`.

## Caveats

- Hyperparameters are tuned on the test images (there are no unseen-class validation images). The cross-fit gives a more honest estimate, but the unseen set is dominated by a few classes (three pairs make up 66.5% of it), so small HM differences are not meaningful.
- Post-hoc calibration cannot beat the closed-world limits (seen 0.685, unseen 0.7735 on B0), which cap HM top-1 at about 0.7265.

# Results: EfficientNet-B0 independent CNN with v2 logit calibration

Test data: DPD-Small, 2,000 seen and 1,448 unseen images (323 seen and 10 unseen crop-disease pairs). Models: two frozen EfficientNet-B0 networks (crop and disease). Only logit handling and inference-time hyperparameters were tuned.

## Files

| File | Contents |
|---|---|
| `metrics_1.json` | The run's main `metrics.json`: uncalibrated, bias-only, v2 held-out and v2 all-test results plus the chosen config. |
| `metrics_calibrated_v2.json` | The chosen config and its metrics (tuned on all test images). |
| `calibration_search_v2.csv` | All 1,008 searched configs, ranked by HM top-1. Top-5 HM is filled in for the best 40 only. |
| `confusion_seen_compositions.npy` | Seen confusion matrix from the calibrated predictions (identical to the `_calibrated` file). |
| `confusion_seen_compositions_calibrated.npy` | Same matrix, saved under an explicit name. |
| `confusion_seen_compositions_uncalibrated.npy` | Seen confusion matrix from the raw composition scores, for comparison. |
| `calibration_curve.png` | Seen versus unseen top-1 trade-off for the **bias-only** sweep (best HM 0.612). The v2 config is not drawn on it. |

Matrix shape is 323 x 333: rows are the 323 true seen pairs, columns are all 333 predicted pairs (first 323 seen, last 10 unseen).

Also produced by the notebook but not copied here: `cross_fit_v2.csv`, `logits_cache.npz`, `predictions.csv`, `top_confusions_seen.csv` and the confusion plots.

## What is new

- **Calibration config:** `tau_c = 0`, `tau_d = 0`, `T_c = 1.0`, `T_d = 1.5`, `tau_pair = 0`, `bias = 4.828`, chosen from 1,008 combinations.
- **Honest estimate:** a 2-fold cross-fit (tune on one half of the test images, score on the other half) is reported alongside the all-test result.
- **Matching confusion matrix:** the seen confusion matrix now comes from the calibrated predictions.
- **Top-5 handling:** near-ties in top-1 HM are broken by top-5 HM.

## Metrics

| Stage | Seen top-1 | Unseen top-1 | HM top-1 | Seen top-5 | Unseen top-5 | HM top-5 |
|---|---|---|---|---|---|---|
| Uncalibrated | 0.6845 | 0.0849 | 0.1511 | 0.8860 | 0.3543 | 0.5062 |
| Bias-only (b = 6.432) | 0.5905 | 0.6347 | 0.6118 | n/a | n/a | n/a |
| v2, tuned on all test images | 0.5950 | 0.7134 | 0.6488 | 0.8650 | 0.9116 | 0.8877 |
| v2, held-out (cross-fit) | 0.5978 | 0.7027 | 0.6459 | 0.8657 | 0.9099 | 0.8872 |

## Differences

| Comparison | Seen top-1 | Unseen top-1 | HM top-1 | Seen top-5 | Unseen top-5 | HM top-5 |
|---|---|---|---|---|---|---|
| v2 vs uncalibrated | -0.0895 | +0.6285 | +0.4977 | -0.0210 | +0.5573 | +0.3815 |
| v2 vs bias-only | +0.0045 | +0.0787 | +0.0370 | n/a | n/a | n/a |
| held-out vs all-test | +0.0028 | -0.0107 | -0.0029 | +0.0007 | -0.0017 | -0.0005 |

## Findings and caveats

- **Prior-based logit adjustment was implemented and searched but not selected.** HM top-1 fell steadily as `tau_d` increased (0.6488 at 0, 0.5207 at 2) and as `tau_pair` increased. The gain over bias-only comes from disease-head temperature scaling (`T_d = 1.5`) together with the unseen bias. The best config with `tau_d > 0` (`tau_d = 0.25`) scores HM top-1 0.6462 and top-5 0.8852, slightly below the selected one.
- **Generalisation:** held-out and all-test HM differ by less than 0.003, so the config does not look overfitted. It is still tuned on test images, so state that in the report.
- **Cost on seen images:** seen images predicted as one of the 10 unseen pairs rise from 2 to 396 (19.8% of the seen test set), which explains the drop in seen top-1 from 0.6845 to 0.5950.
- **Ceiling:** closed-world top-1 is 0.685 (seen) and 0.7735 (unseen), capping HM top-1 at about 0.7265. The held-out HM is about 89% of that.
- **Run-to-run noise:** the uncalibrated and bias-only figures differ from an earlier run in the third decimal (for example bias-only HM 0.6109 then, 0.6118 now). Use these values throughout the report.
- **Top-5 for bias-only** was not recorded, so it is not compared.